# CAREGIVER - Bronze Layer

This notebook ingests the MIMIC-IV `caregiver` source table and stores it as a Bronze Delta table.

A small source check is completed before ingestion to confirm the table structure and caregiver identifier.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"

SOURCE_PATH = "s3://mimic-iii-demo-bucket/mimic-iv-dataset-demo-2.2-folder/icu/caregiver.csv"

caregiver = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SOURCE_PATH)
)

print(f"Rows: {caregiver.count():,}")
print(f"Columns: {len(caregiver.columns)}")

caregiver.printSchema()

### Source Check

`CAREGIVER` contains the unique caregiver identifiers used across the MIMIC-IV ICU data.

Before creating the Bronze table, `caregiver_id` is checked for missing or duplicate values.

In [0]:
# Check caregiver_id
total = caregiver.count()
unique_ids = caregiver.select("caregiver_id").distinct().count()
null_ids = caregiver.filter(F.col("caregiver_id").isNull()).count()

print(f"Rows: {total:,}")
print(f"Unique caregiver_id: {unique_ids:,}")
print(f"Null caregiver_id: {null_ids:,}")
print(f"Duplicate caregiver_id: {total - unique_ids:,}")

### Bronze Ingestion

`caregiver_id` is complete and unique across the source table.

The table is therefore ready to be stored in Bronze with the original structure preserved and basic ingestion metadata added.

In [0]:
# Add ingestion metadata
caregiver_bronze = (
    caregiver
    .withColumn("_source_file", F.lit("caregiver.csv"))
    .withColumn("_source_module", F.lit("icu"))
    .withColumn("_ingestion_timestamp", F.current_timestamp())
)

# Save as Delta table
(
    caregiver_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.caregiver_raw")
)

print("Created: workspace.bronze.caregiver_raw")


In [0]:
source_count = caregiver.count()
bronze_count = spark.table("workspace.bronze.caregiver_raw").count()

print(f"Source: {source_count:,}")
print(f"Bronze: {bronze_count:,}")
print(f"Match: {source_count == bronze_count}")

### Bronze Result

`CAREGIVER` was successfully written to the Bronze layer with all source records preserved.

The `caregiver_id` field is complete and unique, and the Bronze table matches the source row count.